# Pré-processamento

## 1. Importações e Carregamento dos Dados

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib

data_path = Path("../data/creditcard.csv")
df = pd.read_csv(data_path)
print(f"Dataset carregado: {df.shape[0]:,} transações, {df.shape[1]} colunas")

Dataset carregado: 284,807 transações, 31 colunas


## 2. Separação de Features (X) e Target (y)

In [5]:
X = df.drop(columns=["Class"])
y = df["Class"]

print(f"X: {X.shape}  |  y: {y.shape}")
print(f"\nDistribuição das classes:\n{y.value_counts().to_string()}")
print(f"\nProporção de fraudes: {y.mean():.4%}")

X: (284807, 30)  |  y: (284807,)

Distribuição das classes:
Class
0    284315
1       492

Proporção de fraudes: 0.1727%


## 3. Divisão em Treino e Teste (80/20 Estratificado)

A divisão é realizada **antes** da normalização para evitar *data leakage*: o `StandardScaler` será ajustado exclusivamente nos dados de treino e depois aplicado ao conjunto de teste.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino : {X_train.shape[0]:,} amostras ({X_train.shape[0] / len(X):.0%})")
print(f"Teste  : {X_test.shape[0]:,} amostras ({X_test.shape[0] / len(X):.0%})")
print(f"\nDistribuição no treino:\n{y_train.value_counts().to_string()}")
print(f"\nDistribuição no teste:\n{y_test.value_counts().to_string()}")

Treino : 227,845 amostras (80%)
Teste  : 56,962 amostras (20%)

Distribuição no treino:
Class
0    227451
1       394

Distribuição no teste:
Class
0    56864
1       98


## 4. Verificação e Normalização de Escala

As colunas `V1`–`V28` são o resultado de uma **PCA** aplicada no Kaggle antes da publicação do dataset: já estão centradas (média ≈ 0) e portanto **não precisam de normalização adicional**.

`Time` e `Amount` são variáveis brutas e precisam de escalonamento. O `StandardScaler` é ajustado **apenas no conjunto de treino** (`fit_transform`) e depois aplicado ao teste (`transform`), evitando vazamento de informação (*data leakage*).

In [7]:
print("Antes da normalização — Time e Amount (treino):")
print(X_train[["Time", "Amount"]].describe().round(2))

scaler = StandardScaler()
cols_to_scale = ["Time", "Amount"]

X_train = X_train.copy()
X_test = X_test.copy()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print("\nApós normalização — Time e Amount (treino):")
print(X_train[["Time", "Amount"]].describe().round(3))

Antes da normalização — Time e Amount (treino):
            Time     Amount
count  227845.00  227845.00
mean    94885.09      88.18
std     47488.42     250.72
min         0.00       0.00
25%     54228.00       5.64
50%     84805.00      22.00
75%    139364.00      77.49
max    172792.00   25691.16

Após normalização — Time e Amount (treino):
             Time      Amount
count  227845.000  227845.000
mean       -0.000      -0.000
std         1.000       1.000
min        -1.998      -0.352
25%        -0.856      -0.329
50%        -0.212      -0.264
75%         0.937      -0.043
max         1.641     102.117


## 5. Balanceamento das Classes com SMOTE

O SMOTE é aplicado **apenas no conjunto de treino** e será utilizado pelos modelos supervisionados (**Random Forest** e **XGBoost**). Os modelos não supervisionados não dependem de classes balanceadas.

In [8]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

y_train_smote_series = pd.Series(y_train_smote)
print(f"Antes do SMOTE : {y_train.value_counts().to_dict()}")
print(f"Após o SMOTE   : {y_train_smote_series.value_counts().to_dict()}")
print(f"\nX_train_smote shape: {X_train_smote.shape}")

Antes do SMOTE : {0: 227451, 1: 394}
Após o SMOTE   : {0: 227451, 1: 227451}

X_train_smote shape: (454902, 30)


## 6. Preparação para Modelos Não Supervisionados

| Modelo            | Dados de treino   | Estratégia                                                               |
|-------------------|-------------------|--------------------------------------------------------------------------|
| Random Forest     | `X_train_smote`   | Supervisionado; classes balanceadas via SMOTE                            |
| XGBoost           | `X_train_smote`   | Supervisionado; classes balanceadas via SMOTE                            |
| Isolation Forest  | `X_train`         | Não supervisionado — aprende a isolar anomalias pelo caminho de partição |
| Autoencoder (MLP) | `X_train_normal`  | Treinado só em transações normais; fraude = alta reconstrução            |

Todos os modelos são avaliados em `X_test` / `y_test`.

In [9]:
X_train_normal = X_train[y_train == 0]

print(f"X_train        : {X_train.shape}  — Isolation Forest")
print(f"X_train_normal : {X_train_normal.shape}  — Autoencoder (MLPRegressor)")
print(f"X_train_smote  : {X_train_smote.shape}  — Random Forest / XGBoost")
print(f"X_test         : {X_test.shape}  — avaliação de todos os modelos")

X_train        : (227845, 30)  — Isolation Forest
X_train_normal : (227451, 30)  — Autoencoder (MLPRegressor)
X_train_smote  : (454902, 30)  — Random Forest / XGBoost
X_test         : (56962, 30)  — avaliação de todos os modelos


**Considerações sobre o Pré-processamento**

Ao final do pré-processamento temos três versões do conjunto de treino prontas para os 4 modelos:
- `X_train_smote` / `y_train_smote` → Random Forest e XGBoost
- `X_train` (full) → Isolation Forest
- `X_train_normal` (Class=0 apenas) → Autoencoder (MLPRegressor)

# Treinamento dos 3 especialistas

In [10]:
models_path = Path("../models")
models_path.mkdir(exist_ok=True)
print(f"Modelos serão salvos em: {models_path.resolve()}")

Modelos serão salvos em: /home/leandro/programacao/fraud_council/models


## Especialista 1 — Random Forest

Modelo supervisionado que constrói múltiplas árvores de decisão em subconjuntos aleatórios de features e amostras, agregando seus votos por votação majoritária. Robusto a outliers, captura padrões globais não-lineares e é naturalmente interpretável via `feature_importances_` — base dos SHAP values na fase de explicabilidade.

`class_weight="balanced"` adiciona uma segunda camada de proteção contra o desbalanceamento, complementando o SMOTE. Treinado com `X_train_smote`.

In [11]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train_smote, y_train_smote)

rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_pred = rf_model.predict(X_test)

print("=== Random Forest — Conjunto de Teste ===")
print(classification_report(y_test, rf_pred, target_names=["Legítima", "Fraude"]))
print(f"ROC-AUC : {roc_auc_score(y_test, rf_proba):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, rf_proba):.4f}")

joblib.dump(rf_model, models_path / "random_forest.joblib")
print("\nModelo salvo: models/random_forest.joblib")

=== Random Forest — Conjunto de Teste ===
              precision    recall  f1-score   support

    Legítima       1.00      1.00      1.00     56864
      Fraude       0.83      0.84      0.83        98

    accuracy                           1.00     56962
   macro avg       0.91      0.92      0.92     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC : 0.9702
PR-AUC  : 0.8728

Modelo salvo: models/random_forest.joblib


## Especialista 2 — XGBoost

Modelo supervisionado de **gradient boosting** que constrói árvores sequencialmente, cada uma corrigindo os erros da anterior com descida de gradiente. Captura padrões complexos e interações entre features que podem ser invisíveis ao Random Forest. Treinado com `X_train_smote`.

`learning_rate=0.05` e `subsample/colsample_bytree=0.8` adicionam regularização estocástica, reduzindo overfitting. O `eval_metric="aucpr"` prioriza a área sob a curva Precision-Recall — métrica mais informativa que ROC-AUC em datasets altamente desbalanceados como este.

In [15]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    reg_alpha=0.1,
    reg_lambda=0.5
)

xgb_model.fit(X_train_smote, y_train_smote)

xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_pred = xgb_model.predict(X_test)

print("=== XGBoost — Conjunto de Teste ===")
print(classification_report(y_test, xgb_pred, target_names=["Legítima", "Fraude"]))
print(f"ROC-AUC : {roc_auc_score(y_test, xgb_proba):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, xgb_proba):.4f}")

joblib.dump(xgb_model, models_path / "xgboost.joblib")
print("\nModelo salvo: models/xgboost.joblib")

=== XGBoost — Conjunto de Teste ===
              precision    recall  f1-score   support

    Legítima       1.00      1.00      1.00     56864
      Fraude       0.24      0.87      0.38        98

    accuracy                           1.00     56962
   macro avg       0.62      0.93      0.69     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC : 0.9789
PR-AUC  : 0.8424

Modelo salvo: models/xgboost.joblib


## Especialista 3 — Autoencoder (MLPRegressor)

O Autoencoder é treinado **exclusivamente em transações legítimas** (`X_train_normal`). Sua arquitetura encoder-decoder `(30 → 20 → 10 → 20 → 30)` comprime os dados e os reconstrói. Transações fraudulentas, nunca vistas durante o treino, geram um **erro de reconstrução (MSE) maior**.

O score de anomalia é o MSE por transação. O limiar de decisão é o **percentil 99** do MSE calculado sobre o próprio conjunto de treino legítimo, garantindo que apenas as cauda mais anômala seja sinalizada como fraude.

In [19]:
autoencoder = MLPRegressor(
    hidden_layer_sizes=(20, 10, 20),
    activation="tanh",
    solver="adam",
    learning_rate_init=1e-3,
    max_iter=300,
    random_state=42,
    verbose=False,
)

autoencoder.fit(X_train_normal, X_train_normal)

X_train_normal_arr = (
    X_train_normal.values if hasattr(X_train_normal, "values") else X_train_normal
)
X_test_arr = X_test.values if hasattr(X_test, "values") else X_test

train_recon = autoencoder.predict(X_train_normal_arr)
train_mse = np.mean((X_train_normal_arr - train_recon) ** 2, axis=1)
threshold = np.percentile(train_mse, 99)

test_recon = autoencoder.predict(X_test_arr)
ae_scores = np.mean((X_test_arr - test_recon) ** 2, axis=1)
ae_pred = (ae_scores > threshold).astype(int)

print("=== Autoencoder (MLPRegressor) — Conjunto de Teste ===")
print(f"Limiar (percentil 99 do MSE treino): {threshold:.6f}")
print(classification_report(y_test, ae_pred, target_names=["Legítima", "Fraude"]))
print(f"ROC-AUC : {roc_auc_score(y_test, ae_scores):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, ae_scores):.4f}")

joblib.dump(autoencoder, models_path / "autoencoder.joblib")
np.save(models_path / "autoencoder_threshold.npy", threshold)
print("\nModelo salvo : models/autoencoder.joblib")
print(f"Limiar salvo : models/autoencoder_threshold.npy  (threshold={threshold:.6f})")

/home/leandro/programacao/fraud_council/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but MLPRegressor was fitted with feature names
  warnings.warn(


=== Autoencoder (MLPRegressor) — Conjunto de Teste ===
Limiar (percentil 99 do MSE treino): 0.849289
              precision    recall  f1-score   support

    Legítima       1.00      0.99      0.99     56864
      Fraude       0.12      0.85      0.22        98

    accuracy                           0.99     56962
   macro avg       0.56      0.92      0.61     56962
weighted avg       1.00      0.99      0.99     56962

ROC-AUC : 0.9507
PR-AUC  : 0.5517

Modelo salvo : models/autoencoder.joblib
Limiar salvo : models/autoencoder_threshold.npy  (threshold=0.849289)


/home/leandro/programacao/fraud_council/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but MLPRegressor was fitted with feature names
  warnings.warn(
